## Imports

In [387]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import re  

## Load + Preview

In [388]:
# Load the combined dataset
df = pd.read_csv("data/all_fighter_records.csv")

# Display the first 5 rows
df.head()

# Get basic info about columns and data types
df.info()

# Check for missing values
df.isnull().sum()

# Unique fighters
df["Name"].value_counts()

# Most common opponents
df["Opponent"].value_counts().head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2295 entries, 0 to 2294
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Name            2295 non-null   object
 1   Height          2140 non-null   object
 2   Reach           2140 non-null   object
 3   Stance          2096 non-null   object
 4   Weight          2140 non-null   object
 5   Opponent        2295 non-null   object
 6   No.             2295 non-null   int64 
 7   Result          2295 non-null   object
 8   Record          2295 non-null   object
 9   Type            2295 non-null   object
 10  Round, time     2258 non-null   object
 11  Date            2295 non-null   object
 12  Location        2295 non-null   object
 13  Notes           1082 non-null   object
 14  Age             780 non-null    object
 15  Round(s), time  37 non-null     object
dtypes: int64(1), object(15)
memory usage: 287.0+ KB


Opponent
Manny Pacquiao          9
Oscar De La Hoya        7
Canelo Álvarez          7
Floyd Mayweather Jr.    7
Héctor Camacho          6
Tyson Fury              6
Sugar Ray Leonard       6
Evander Holyfield       5
Roberto Durán           5
Shane Mosley            5
Name: count, dtype: int64

## Cleaning

In [389]:
# Create 'is_title_fight' column BEFORE dropping 'Notes'
df["is_title_fight"] = df["Notes"].notna().astype(int)

# Remove specified columns
cols_to_drop = ["Notes", "Age", "Round(s), time", "Weight", "Location"]
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns], errors="ignore")

# Count missing values after removing those columns
na_counts = df.isna().sum()

# Display columns with any missing values
print("Missing values per column:")
print(na_counts[na_counts > 0])

# Drop rows with any missing values
df = df.dropna()

# Convert date column to datetime and sort by fighter and date
df["Date"] = pd.to_datetime(df["Date"], format='mixed', dayfirst=True, errors='coerce')
df = df.sort_values(by=["Name", "Date"]).reset_index(drop=True)

# Confirm shape and preview cleaned data
print(f"\nDataset shape after cleaning: {df.shape}")
df.head()

Missing values per column:
Height         155
Reach          155
Stance         199
Round, time     37
dtype: int64

Dataset shape after cleaning: (2059, 12)


,Name,Height,Reach,Stance,Opponent,No.,Result,Record,Type,"Round, time",Date,is_title_fight
0,Andre Ward,6 ft 0 in (183 cm)[1],71 in (180 cm),Orthodox,Chris Molina,1,Win,1–0,TKO,"2 (4), 0:40",2004-12-18,0
1,Andre Ward,6 ft 0 in (183 cm)[1],71 in (180 cm),Orthodox,Kenny Kost,2,Win,2–0,UD,6,2005-02-10,0
2,Andre Ward,6 ft 0 in (183 cm)[1],71 in (180 cm),Orthodox,Roy Ashworth,3,Win,3–0,DQ,"3 (6), 2:56",2005-04-07,1
3,Andre Ward,6 ft 0 in (183 cm)[1],71 in (180 cm),Orthodox,Ben Aragon,4,Win,4–0,TKO,"3 (6), 0:59",2005-06-18,0
4,Andre Ward,6 ft 0 in (183 cm)[1],71 in (180 cm),Orthodox,Christopher Holt,5,Win,5–0,RTD,"3 (6), 3:00",2005-08-18,0


## Feature Extraction

In [390]:
# Extract numeric cm from Height
def extract_height_cm(text):
    if pd.isna(text):
        return None
    match = re.search(r"\(?(\d+)\s?cm", text)  # Fixed regex pattern
    return float(match.group(1)) if match else None

# Extract numeric cm from Reach
def extract_reach_cm(text):
    if pd.isna(text):
        return None
    match = re.search(r"\(?(\d+)\s?cm", text)  # Fixed regex pattern
    return int(match.group(1)) if match else None

# Convert time to seconds (e.g., "2:30" -> 150)
def time_to_seconds(time_str):
    if pd.isna(time_str) or time_str is None:
        return 180  # Default to 3 minutes (full round)
    
    parts = time_str.split(':')
    if len(parts) == 2:
        minutes, seconds = int(parts[0]), int(parts[1])
        return minutes * 60 + seconds
    else:
        # Handle edge cases
        return 180  # Default to full round if format not recognized

# Clean "Round, time" into Round_ended and Time_ended_seconds
def split_round_time(rt):
    if pd.isna(rt):
        return (12, 180)  # Default to round 12, full round (3 minutes)
    
    match = re.search(r"(\d+)[^,]*,\s*([\d:]+)", str(rt))  # Fixed regex pattern
    if match:
        round_num = float(match.group(1))
        time_str = match.group(2)
        time_secs = time_to_seconds(time_str)
        return (round_num, time_secs)
    
    return (12, 180)  # Default to round 12, full round

# Apply functions
df["Height_cm"] = df["Height"].apply(extract_height_cm)
df["Reach_cm"] = df["Reach"].apply(extract_reach_cm)
df[["Round_ended", "Time_ended_seconds"]] = df["Round, time"].apply(lambda x: pd.Series(split_round_time(x)))

# Fill any remaining NaN values
df["Round_ended"] = df["Round_ended"].fillna(12)
df["Time_ended_seconds"] = df["Time_ended_seconds"].fillna(180)  # 3 minutes in seconds

# Check the results
print(f"Missing Height_cm values: {df['Height_cm'].isna().sum()}")
print(f"Missing Reach_cm values: {df['Reach_cm'].isna().sum()}")
print(f"Missing Round_ended values: {df['Round_ended'].isna().sum()}")
print(f"Missing Time_ended_seconds values: {df['Time_ended_seconds'].isna().sum()}")

# Display the first few rows
df.head()

Missing Height_cm values: 139
Missing Reach_cm values: 0
Missing Round_ended values: 0
Missing Time_ended_seconds values: 0


,Name,Height,Reach,Stance,Opponent,No.,Result,Record,Type,"Round, time",Date,is_title_fight,Height_cm,Reach_cm,Round_ended,Time_ended_seconds
0,Andre Ward,6 ft 0 in (183 cm)[1],71 in (180 cm),Orthodox,Chris Molina,1,Win,1–0,TKO,"2 (4), 0:40",2004-12-18,0,183.0,180,2.0,40.0
1,Andre Ward,6 ft 0 in (183 cm)[1],71 in (180 cm),Orthodox,Kenny Kost,2,Win,2–0,UD,6,2005-02-10,0,183.0,180,12.0,180.0
2,Andre Ward,6 ft 0 in (183 cm)[1],71 in (180 cm),Orthodox,Roy Ashworth,3,Win,3–0,DQ,"3 (6), 2:56",2005-04-07,1,183.0,180,3.0,176.0
3,Andre Ward,6 ft 0 in (183 cm)[1],71 in (180 cm),Orthodox,Ben Aragon,4,Win,4–0,TKO,"3 (6), 0:59",2005-06-18,0,183.0,180,3.0,59.0
4,Andre Ward,6 ft 0 in (183 cm)[1],71 in (180 cm),Orthodox,Christopher Holt,5,Win,5–0,RTD,"3 (6), 3:00",2005-08-18,0,183.0,180,3.0,180.0


## Advanced Features

In [391]:
# Clean stance
df["Stance"] = df["Stance"].str.strip().str.title()
df["is_orthodox"] = ((df["Stance"] == "Orthodox") | (df["Stance"] == "Other stance")).astype(int)
df["is_southpaw"] = ((df["Stance"] == "Southpaw") | (df["Stance"] == "Other stance")).astype(int)


# Win and KO flags
df["is_win"] = df["Result"].str.lower().str.strip() == "win"
df["is_ko"] = df["Type"].str.upper().str.strip().isin(["KO", "TKO"])

# Decision-based flags
df["Outcome"] = df["Type"].str.upper().str.strip()
df["is_decision"] = df["Outcome"].str.contains("DECISION|UD|SD|MD")
df["is_UD"] = (df["Outcome"] == "UD").astype(int)

# Calculate days since last fight (time-based feature)
df["prev_fight_date"] = df.groupby("Name")["Date"].shift(1)
df["days_since_last_fight"] = (df["Date"] - df["prev_fight_date"]).dt.days

# Aggregate win % and KO rate
summary = df.groupby("Name").agg(
    total_fights=("is_win", "count"),
    wins=("is_win", "sum"),
    kos=("is_ko", "sum")
).reset_index()
summary["win_percent"] = (summary["wins"] / summary["total_fights"]).round(3)
summary["ko_rate"] = (summary["kos"] / summary["wins"]).round(3)

df = df.merge(summary[["Name", "win_percent", "ko_rate"]], on="Name", how="left")

# Final cleanup: drop unneeded columns
cols_to_drop = ["Height", "Reach", "Stance", "No.", "Type", "Round, time", "Time_ended"]
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns], errors="ignore")

# Preview final dataset with all features
pd.set_option("display.max_columns", None)
df.head()

,Name,Opponent,Result,Record,Date,is_title_fight,Height_cm,Reach_cm,Round_ended,Time_ended_seconds,is_orthodox,is_southpaw,is_win,is_ko,Outcome,is_decision,is_UD,prev_fight_date,days_since_last_fight,win_percent,ko_rate
0,Andre Ward,Chris Molina,Win,1–0,2004-12-18,0,183.0,180,2.0,40.0,1,0,True,True,TKO,False,0,NaT,NaN,1.0,0.438
1,Andre Ward,Kenny Kost,Win,2–0,2005-02-10,0,183.0,180,12.0,180.0,1,0,True,False,UD,True,1,2004-12-18,54.0,1.0,0.438
2,Andre Ward,Roy Ashworth,Win,3–0,2005-04-07,1,183.0,180,3.0,176.0,1,0,True,False,DQ,False,0,2005-02-10,56.0,1.0,0.438
3,Andre Ward,Ben Aragon,Win,4–0,2005-06-18,0,183.0,180,3.0,59.0,1,0,True,True,TKO,False,0,2005-04-07,72.0,1.0,0.438
4,Andre Ward,Christopher Holt,Win,5–0,2005-08-18,0,183.0,180,3.0,180.0,1,0,True,False,RTD,False,0,2005-06-18,61.0,1.0,0.438


## ELO

In [392]:
# Start with a fresh dict of fighter ELOs
fighter_elos = {}
elo_histories = []

# Parameters
K = 32
DEFAULT_ELO = 1500
KO_BONUS = 5
UD_BONUS = 2

# Loop through fights
for _, row in df.iterrows():
    fighter = row["Name"]
    opponent = row["Opponent"]
    result = int(row["is_win"])
    is_ko = int(row.get("is_ko", 0))
    is_ud = int(row.get("is_UD", 0))

    # Get current ELOs
    elo_fighter = fighter_elos.get(fighter, DEFAULT_ELO)
    elo_opponent = fighter_elos.get(opponent, DEFAULT_ELO)

    # Store ELO snapshot for feature engineering
    elo_snapshot = (elo_fighter, elo_opponent)
    
    # Expected score
    expected_score = 1 / (1 + 10 ** ((elo_opponent - elo_fighter) / 400))

    # Base ELO update
    new_elo_fighter = elo_fighter + K * (result - expected_score)

    # Apply bonus if win
    if result == 1:
        new_elo_fighter += KO_BONUS * is_ko + UD_BONUS * is_ud

    # Save updated ELO
    fighter_elos[fighter] = new_elo_fighter

    # Optional: update opponent
    if opponent in df["Name"].values:
        new_elo_opponent = elo_opponent + K * ((1 - result) - (1 - expected_score))
        fighter_elos[opponent] = new_elo_opponent

    # Track history
    elo_histories.append(new_elo_fighter)
    
# Add ELO features to DataFrame
df["elo_progression"] = elo_histories

# Add fighter and opponent ELO at the time of each fight
fighter_elos_snapshot = []
for _, row in df.iterrows():
    fighter = row["Name"]
    opponent = row["Opponent"]
    fighter_elos_snapshot.append((fighter_elos.get(fighter, DEFAULT_ELO), fighter_elos.get(opponent, DEFAULT_ELO)))
df[["A_elo", "B_elo"]] = pd.DataFrame(fighter_elos_snapshot, index=df.index)

In [393]:
# Get last fight per fighter
last_fights = df.groupby("Name").tail(1)[["Name"]].copy()

# Attach latest ELOs from the dictionary
last_fights["ELO"] = last_fights["Name"].map(fighter_elos)

# Sort by ELO descending
last_fights = last_fights.sort_values(by="ELO", ascending=False).reset_index(drop=True)

# Display top fighters by ELO rating
last_fights

,Name,ELO
0,Carlos Monzón,2027.341728
1,George Foreman,1993.101552
2,Julio César Chávez,1989.009849
3,Marvin Hagler,1985.669011
4,Wladimir Klitschko,1955.033674
5,Thomas Hearns,1954.474653
6,Terence Crawford,1949.749981
7,Canelo Álvarez,1941.743701
8,Vitali Klitschko,1936.036178
9,Lennox Lewis,1921.577960


## Dataset for modelling

In [394]:
def prepare_fight_data(df, fighter_elos):
    # Create features for both fighters
    fight_data = []
    
    # Get all unique fighters
    unique_fighters = set(df['Name'].unique())
    
    # For each fight in the dataset
    for _, row in df.iterrows():
        fighter = row['Name']
        opponent = row['Opponent']
        
        # Only use fights where both fighters are in our dataset
        if opponent in unique_fighters:
            # Get fighter stats
            fighter_height = row['Height_cm']
            fighter_reach = row['Reach_cm']
            fighter_orthodox = row['is_orthodox']
            fighter_southpaw = row['is_southpaw']
            fighter_win_pct = row['win_percent']
            fighter_ko_rate = row['ko_rate']
            fighter_elo = row['A_elo']
            
            # Get opponent stats - find their most recent fight before this one
            opp_fights = df[(df['Name'] == opponent) & (df['Date'] < row['Date'])]
            
            if len(opp_fights) > 0:
                opp_last_fight = opp_fights.iloc[-1]
                opp_height = opp_last_fight['Height_cm']
                opp_reach = opp_last_fight['Reach_cm']
                opp_orthodox = opp_last_fight['is_orthodox']
                opp_southpaw = opp_last_fight['is_southpaw']
                opp_win_pct = opp_last_fight['win_percent']
                opp_ko_rate = opp_last_fight['ko_rate']
                opp_elo = row['B_elo']
                
                # Skip if any essential data is missing
                if (fighter_height is None or opp_height is None or
                    fighter_reach is None or opp_reach is None):
                    continue
                
                # Calculate differentials
                height_diff = fighter_height - opp_height
                reach_diff = fighter_reach - opp_reach
                elo_diff = fighter_elo - opp_elo
                
                # Feature for title fight
                is_title = row['is_title_fight']
                
                # Days since last fight
                days_inactive = row['days_since_last_fight'] if not pd.isna(row['days_since_last_fight']) else 180  # Default to 6 months
                
                # Target variables
                result = 1 if row['is_win'] else 0
                ko_win = 1 if row['is_win'] and row['is_ko'] else 0
                decision_win = 1 if row['is_win'] and row['is_decision'] else 0
                round_ended = row['Round_ended'] if not pd.isna(row['Round_ended']) else 12  # Default to going the distance
                
                # Create feature vector
                features = [
                    fighter_height, fighter_reach, fighter_orthodox, fighter_southpaw,
                    fighter_win_pct, fighter_ko_rate, fighter_elo,
                    opp_height, opp_reach, opp_orthodox, opp_southpaw,
                    opp_win_pct, opp_ko_rate, opp_elo,
                    height_diff, reach_diff, elo_diff, is_title, days_inactive
                ]
                
                # Results
                targets = [result, ko_win, decision_win, round_ended]
                
                fight_data.append(features + targets)
    
    # Handle case where no valid fights were found
    if not fight_data:
        print("Warning: No valid fight data could be prepared. Check for missing values in height/reach.")
        return None
        
    # Convert to DataFrame
    columns = [
        'fighter_height', 'fighter_reach', 'fighter_orthodox', 'fighter_southpaw',
        'fighter_win_pct', 'fighter_ko_rate', 'fighter_elo',
        'opp_height', 'opp_reach', 'opp_orthodox', 'opp_southpaw',
        'opp_win_pct', 'opp_ko_rate', 'opp_elo',
        'height_diff', 'reach_diff', 'elo_diff', 'is_title', 'days_inactive',
        'result', 'ko_win', 'decision_win', 'round_ended'
    ]
    
    fight_df = pd.DataFrame(fight_data, columns=columns)
    return fight_df

In [395]:
# Prepare dataset
fight_df = prepare_fight_data(df, fighter_elos)
fight_df.head()

,fighter_height,fighter_reach,fighter_orthodox,fighter_southpaw,fighter_win_pct,fighter_ko_rate,fighter_elo,opp_height,opp_reach,opp_orthodox,opp_southpaw,opp_win_pct,opp_ko_rate,opp_elo,height_diff,reach_diff,elo_diff,is_title,days_inactive,result,ko_win,decision_win,round_ended
0,198.0,208,1,0,0.875,0.929,1751.651338,NaN,206,1,0,0.928,0.812,1955.033674,NaN,2,-203.382336,1,140.0,1,1,0,11.0
1,198.0,208,1,0,0.875,0.929,1751.651338,NaN,198,0,1,1.000,0.565,1868.758788,NaN,10,-117.107449,1,287.0,0,0,0,12.0
2,198.0,208,1,0,0.875,0.929,1751.651338,NaN,198,0,1,1.000,0.565,1868.758788,NaN,10,-117.107449,1,329.0,0,0,0,12.0
3,182.0,185,1,0,0.955,0.905,1770.447535,183.0,183,1,0,0.960,0.500,1826.317303,-1.0,2,-55.869768,1,273.0,1,0,1,12.0
4,182.0,185,1,0,0.955,0.905,1770.447535,183.0,183,1,0,0.960,0.500,1826.317303,-1.0,2,-55.869768,1,133.0,0,0,0,12.0


In [396]:
# Check which columns have NaN values
nan_counts = fight_df.isna().sum()
print("NaN values in each column:")
print(nan_counts[nan_counts > 0])

print("NaN values in each column:")
fight_df = fight_df.dropna()
nan_counts = fight_df.isna().sum()
print(nan_counts[nan_counts > 0])

NaN values in each column:
fighter_height     7
opp_height         7
height_diff       14
dtype: int64
NaN values in each column:
Series([], dtype: int64)


## Model and Prediction

In [397]:
## Model and Prediction

# Split into features and targets
X = fight_df.drop(['result', 'ko_win', 'decision_win', 'round_ended'], axis=1)
y_win = fight_df['result']
y_ko = fight_df['ko_win']
y_decision = fight_df['decision_win']
y_round = fight_df['round_ended']

# Train-test split
X_train, X_test, y_win_train, y_win_test = train_test_split(
    X, y_win, test_size=0.25, random_state=42
)

# Create and train the win probability model
win_model = RandomForestClassifier(n_estimators=100, random_state=42)
win_model.fit(X_train, y_win_train)

# Evaluate the win prediction model
win_preds = win_model.predict(X_test)
win_accuracy = accuracy_score(y_win_test, win_preds)
print(f"Win prediction accuracy: {win_accuracy:.4f}")
print("\nWin prediction classification report:")
print(classification_report(y_win_test, win_preds))

# Feature importance for win model
win_feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': win_model.feature_importances_
}).sort_values('Importance', ascending=False)
print("\nTop features for win prediction:")
print(win_feature_importance.head(10))

# Now train models for KO prediction
X_train, X_test, y_ko_train, y_ko_test = train_test_split(
    X, y_ko, test_size=0.25, random_state=42
)
ko_model = RandomForestClassifier(n_estimators=100, random_state=42)
ko_model.fit(X_train, y_ko_train)

# Evaluate the KO prediction model
ko_preds = ko_model.predict(X_test)
ko_accuracy = accuracy_score(y_ko_test, ko_preds)
print(f"\nKO prediction accuracy: {ko_accuracy:.4f}")

# Train model for decision prediction
X_train, X_test, y_decision_train, y_decision_test = train_test_split(
    X, y_decision, test_size=0.25, random_state=42
)
decision_model = RandomForestClassifier(n_estimators=100, random_state=42)
decision_model.fit(X_train, y_decision_train)

# Train model for round prediction
X_train, X_test, y_round_train, y_round_test = train_test_split(
    X, y_round, test_size=0.25, random_state=42
)
round_model = RandomForestClassifier(n_estimators=100, random_state=42)
round_model.fit(X_train, y_round_train)

## FIGHT Prediction Function

def predict_fight(fighter1_name, fighter2_name, df, fighter_elos, models):
    """
    Predict the outcome of a fight between two fighters.
    
    Parameters:
    -----------
    fighter1_name : str
        Name of the first fighter
    fighter2_name : str
        Name of the second fighter
    df : DataFrame
        The original cleaned dataset
    fighter_elos : dict
        Dictionary of fighter ELO ratings
    models : tuple
        Tuple of (win_model, ko_model, decision_model, round_model)
        
    Returns:
    --------
    dict
        Dictionary containing prediction results
    """
    win_model, ko_model, decision_model, round_model = models
    
    # Get the latest data for each fighter
    fighter1_data = df[df['Name'] == fighter1_name].iloc[-1] if len(df[df['Name'] == fighter1_name]) > 0 else None
    fighter2_data = df[df['Name'] == fighter2_name].iloc[-1] if len(df[df['Name'] == fighter2_name]) > 0 else None
    
    if fighter1_data is None or fighter2_data is None:
        return {"error": f"Could not find data for {'fighter1' if fighter1_data is None else 'fighter2'}"}
    
    # Extract fighter attributes
    fighter1_height = fighter1_data['Height_cm']
    fighter1_reach = fighter1_data['Reach_cm']
    fighter1_orthodox = fighter1_data['is_orthodox']
    fighter1_southpaw = fighter1_data['is_southpaw']
    fighter1_win_pct = fighter1_data['win_percent']
    fighter1_ko_rate = fighter1_data['ko_rate']
    fighter1_elo = fighter_elos.get(fighter1_name, 1500)
    
    fighter2_height = fighter2_data['Height_cm']
    fighter2_reach = fighter2_data['Reach_cm']
    fighter2_orthodox = fighter2_data['is_orthodox']
    fighter2_southpaw = fighter2_data['is_southpaw']
    fighter2_win_pct = fighter2_data['win_percent']
    fighter2_ko_rate = fighter2_data['ko_rate']
    fighter2_elo = fighter_elos.get(fighter2_name, 1500)
    
    # Calculate differentials
    height_diff = fighter1_height - fighter2_height
    reach_diff = fighter1_reach - fighter2_reach
    elo_diff = fighter1_elo - fighter2_elo
    
    # Default values
    is_title = 0  # Not a title fight by default
    days_inactive = 90  # Default to 3 months of inactivity
    
    # Create feature vector - assuming fighter1 is 'A' and fighter2 is 'B'
    features_a_vs_b = [
        fighter1_height, fighter1_reach, fighter1_orthodox, fighter1_southpaw,
        fighter1_win_pct, fighter1_ko_rate, fighter1_elo,
        fighter2_height, fighter2_reach, fighter2_orthodox, fighter2_southpaw,
        fighter2_win_pct, fighter2_ko_rate, fighter2_elo,
        height_diff, reach_diff, elo_diff, is_title, days_inactive
    ]
    
    # Reverse feature vector - assuming fighter2 is 'A' and fighter1 is 'B'
    features_b_vs_a = [
        fighter2_height, fighter2_reach, fighter2_orthodox, fighter2_southpaw,
        fighter2_win_pct, fighter2_ko_rate, fighter2_elo,
        fighter1_height, fighter1_reach, fighter1_orthodox, fighter1_southpaw,
        fighter1_win_pct, fighter1_ko_rate, fighter1_elo,
        -height_diff, -reach_diff, -elo_diff, is_title, days_inactive
    ]
    
    # Make predictions for both scenarios
    # Convert to DataFrame to match expected input format
    features_a_vs_b_df = pd.DataFrame([features_a_vs_b], columns=X.columns)
    features_b_vs_a_df = pd.DataFrame([features_b_vs_a], columns=X.columns)
    
    # Win probability
    win_prob_a = win_model.predict_proba(features_a_vs_b_df)[0][1]
    win_prob_b = win_model.predict_proba(features_b_vs_a_df)[0][1]
    
    # KO probability if win
    ko_prob_a = ko_model.predict_proba(features_a_vs_b_df)[0][1] if win_prob_a > 0.5 else 0
    ko_prob_b = ko_model.predict_proba(features_b_vs_a_df)[0][1] if win_prob_b > 0.5 else 0
    
    # Decision probability if win
    decision_prob_a = decision_model.predict_proba(features_a_vs_b_df)[0][1] if win_prob_a > 0.5 else 0
    decision_prob_b = decision_model.predict_proba(features_b_vs_a_df)[0][1] if win_prob_b > 0.5 else 0
    
    # Round prediction if KO
    round_pred_a = round_model.predict(features_a_vs_b_df)[0] if ko_prob_a > 0.5 else 12
    round_pred_b = round_model.predict(features_b_vs_a_df)[0] if ko_prob_b > 0.5 else 12
    
    # Determine the winner
    # Normalize win probabilities to ensure they sum to 1
    total_prob = win_prob_a + win_prob_b
    if total_prob > 0:  # Avoid division by zero
        win_prob_a_norm = win_prob_a / total_prob
        win_prob_b_norm = win_prob_b / total_prob
    else:
        win_prob_a_norm = 0.5
        win_prob_b_norm = 0.5
    
    if win_prob_a_norm > win_prob_b_norm:
        winner = fighter1_name
        loser = fighter2_name
        win_prob = win_prob_a_norm
        ko_prob = ko_prob_a
        decision_prob = decision_prob_a
        round_pred = round_pred_a
    else:
        winner = fighter2_name
        loser = fighter1_name
        win_prob = win_prob_b_norm
        ko_prob = ko_prob_b
        decision_prob = decision_prob_b
        round_pred = round_pred_b
    
    # Determine the method of victory
    if ko_prob > decision_prob:
        method = "KO/TKO"
        round_str = f"Round {int(round_pred)}"
    else:
        method = "Decision"
        round_str = "12 Rounds"
    
    # Create detailed results dictionary
    results = {
        "fighter1": fighter1_name,
        "fighter2": fighter2_name,
        "fighter1_elo": fighter1_elo,
        "fighter2_elo": fighter2_elo,
        "fighter1_win_prob": win_prob_a,
        "fighter2_win_prob": win_prob_b,
        "winner": winner,
        "loser": loser,
        "win_probability": win_prob,
        "method": method,
        "round": round_str,
        "fighter1_attrs": {
            "height": fighter1_height,
            "reach": fighter1_reach,
            "stance": "Orthodox" if fighter1_orthodox else "Southpaw" if fighter1_southpaw else "Other",
            "win_pct": fighter1_win_pct,
            "ko_rate": fighter1_ko_rate
        },
        "fighter2_attrs": {
            "height": fighter2_height,
            "reach": fighter2_reach,
            "stance": "Orthodox" if fighter2_orthodox else "Southpaw" if fighter2_southpaw else "Other", 
            "win_pct": fighter2_win_pct,
            "ko_rate": fighter2_ko_rate
        }
    }
    
    return results

# Let's create a function to print the fight prediction in a nice format
def print_fight_prediction(prediction):
    """Print a nicely formatted fight prediction"""
    if "error" in prediction:
        print(f"Error: {prediction['error']}")
        return
    
    print("\n" + "="*60)
    print(f"🥊  FIGHT PREDICTION: {prediction['fighter1']} vs {prediction['fighter2']}  🥊")
    print("="*60)
    
    print(f"\n⚔️  {prediction['fighter1']} (ELO: {prediction['fighter1_elo']:.0f}) vs {prediction['fighter2']} (ELO: {prediction['fighter2_elo']:.0f})")
    
    print(f"\n📊 Fighter Attributes:")
    print(f"   {prediction['fighter1']}: {prediction['fighter1_attrs']['height']}cm, {prediction['fighter1_attrs']['reach']}cm reach, {prediction['fighter1_attrs']['stance']} stance")
    print(f"   {prediction['fighter1']} stats: {prediction['fighter1_attrs']['win_pct']*100:.1f}% win rate, {prediction['fighter1_attrs']['ko_rate']*100:.1f}% KO rate")
    print(f"   {prediction['fighter2']}: {prediction['fighter2_attrs']['height']}cm, {prediction['fighter2_attrs']['reach']}cm reach, {prediction['fighter2_attrs']['stance']} stance")
    print(f"   {prediction['fighter2']} stats: {prediction['fighter2_attrs']['win_pct']*100:.1f}% win rate, {prediction['fighter2_attrs']['ko_rate']*100:.1f}% KO rate")
    
    print(f"\n🥇 WINNER: {prediction['winner']}")
    print(f"   Win probability: {prediction['win_probability']*100:.1f}%")
    print(f"   Method: {prediction['method']} in {prediction['round']}")
    
    print("\n" + "="*60)

# Example usage:
# Prepare models for prediction
models = (win_model, ko_model, decision_model, round_model)

# Example fight
fighter1 = "Naoya Inoue"  # Replace with actual fighter name from dataset
fighter2 = "Zab Judah"      # Replace with actual fighter name from dataset

# Predict the fight
prediction = predict_fight(fighter1, fighter2, df, fighter_elos, models)
print_fight_prediction(prediction)

Win prediction accuracy: 0.7727

Win prediction classification report:
              precision    recall  f1-score   support

           0       0.79      0.85      0.81        13
           1       0.75      0.67      0.71         9

    accuracy                           0.77        22
   macro avg       0.77      0.76      0.76        22
weighted avg       0.77      0.77      0.77        22


Top features for win prediction:
            Feature  Importance
18    days_inactive    0.134702
12      opp_ko_rate    0.114024
16         elo_diff    0.091291
11      opp_win_pct    0.087793
14      height_diff    0.087501
5   fighter_ko_rate    0.068961
15       reach_diff    0.066162
13          opp_elo    0.058316
7        opp_height    0.055486
4   fighter_win_pct    0.054343

KO prediction accuracy: 0.7727

🥊  FIGHT PREDICTION: Naoya Inoue vs Zab Judah  🥊

⚔️  Naoya Inoue (ELO: 1891) vs Zab Judah (ELO: 1701)

📊 Fighter Attributes:
   Naoya Inoue: 165.0cm, 171cm reach, Orthodox stance
   